# Fabric Stain Locked Image-Level Evaluation & Defect Segmentation (V8 Protocol)

This V8 notebook implements the next-generation fabric stain anomaly detection and segmentation pipeline:

1. Quad-Scale Hierarchical Reflection Detrending:
   - Nano-kernel (k=15): Sharply isolates micro-droplets and fine thread flaws without over-smoothing.
   - Micro-kernel (k=31): Preserves localized droplet stains and splatter patterns.
   - Meso-kernel (k=61): Standard chemical drop stains.
   - Macro-kernel (k=101): Broad diffuse watermarks and large faint smudges.
2. Border Fringe Attenuation (5-pixel soft cosine taper):
   - Eliminates edge cut-fiber noise and boundary artifacts that caused false seeds on image perimeters.
3. Multi-Resolution Anomaly Fusion & Normalization:
   - Robust Median/IQR background normalization eliminates baseline weave variance across fabric batches.
4. Dual-Stream Defect Clustering with Primary & Shadow Crease Disambiguation:
   - Calibrated seed integration: tau_seed=13.5, tau_low=3.3.
   - Secondary Shadow & Primary Crease Classifier: Isolates both high-aspect primary fold creases (axis_ratio >= 3.5, span >= 50) and secondary parallel shadow bands.
   - Intensity-Bounded Protection: Any cluster with peak > 35.0 or mean > 15.0 is strictly protected as a Chemical Stain.
5. Organic Mask Consolidation:
   - Morphological closing and hole-filling ensure contiguous, natural defect bodies that cover the full physical stain.
6. Production-Grade Visual Outputs:
   - 6-Panel Diagnostic Previews + Color Composite Overlays with Bounding Boxes (Amber for stains, Cyan for creases).
   - Full serialization to test_maps_v8_float16.npz and fabric_stain_evaluation_v8_results.zip.


In [ ]:
from pathlib import Path
import csv, importlib.util, io, json, random, shutil, subprocess, sys, time, zipfile

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/labelinspect')
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK / 'fabric_stain_evaluation_v8'
OUTPUT.mkdir(parents=True, exist_ok=True)

def find_dataset(search_root):
    found = []
    for path in search_root.rglob('fabric_stain_pilot'):
        if path.is_dir() and (path / 'val_normal').is_dir() and (path / 'test' / 'stain').is_dir():
            found.append(path)
    return sorted(set(found))

dataset_candidates = find_dataset(KAGGLE_INPUT)
if not dataset_candidates:
    matching = []
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        with zipfile.ZipFile(archive_path) as archive:
            names = ['/' + item.filename.replace('\\', '/').lstrip('/') for item in archive.infolist()]
            if any('/fabric_stain_pilot/val_normal/' in name for name in names):
                matching.append(archive_path)
    if len(matching) == 1:
        extraction_root = WORK / 'uploaded_data'
        extraction_root.mkdir(parents=True, exist_ok=True)
        resolved = extraction_root.resolve()
        with zipfile.ZipFile(matching[0]) as archive:
            for item in archive.infolist():
                target = (extraction_root / item.filename).resolve()
                if target != resolved and resolved not in target.parents:
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        dataset_candidates = find_dataset(extraction_root)
if len(dataset_candidates) != 1:
    raise FileNotFoundError('Expected one fabric_stain_pilot dataset: ' + repr([str(p) for p in dataset_candidates]))
DATA_ROOT = dataset_candidates[0]

all_pt_files = sorted(KAGGLE_INPUT.rglob('*.pt'))
if not all_pt_files:
    extracted_roots = []
    for data_pickle in KAGGLE_INPUT.rglob('data.pkl'):
        candidate = data_pickle.parent
        if (candidate / 'data').is_dir() and (candidate / 'version').is_file():
            extracted_roots.append(candidate)
    extracted_roots = sorted(set(extracted_roots))
    if len(extracted_roots) == 1:
        archive_root = extracted_roots[0]
        rebuilt = WORK / 'rebuilt_fabric_stain_checkpoint.pt'
        with zipfile.ZipFile(rebuilt, 'w', compression=zipfile.ZIP_STORED) as archive:
            for source_file in sorted(archive_root.rglob('*')):
                if source_file.is_file():
                    archive.write(source_file, f'{archive_root.name}/{source_file.relative_to(archive_root).as_posix()}')
        all_pt_files = [rebuilt]
        print('Rebuilt Kaggle-extracted checkpoint:', rebuilt)
if len(all_pt_files) != 1:
    raise FileNotFoundError('Expected one checkpoint input; found: ' + repr([str(p) for p in all_pt_files]))
CHECKPOINT = all_pt_files[0]

SEED = 230224
IMAGE_SIZE = 224
BATCH_SIZE = 4
T_DISTANCE = 50
NUM_DIFFUSION_SAMPLES = 2
DETREND_K1 = 15   # Nano: micro-droplets, fine thread flaws
DETREND_K2 = 31   # Micro: localized drops, splatter
DETREND_K3 = 61   # Meso: typical chemical stains
DETREND_K4 = 101  # Macro: broad diffuse watermarks
SMOOTH_KERNEL = 5
BORDER_TAPER_PIXELS = 5
MIN_CLUSTER_AREA = 25
DIFFUSE_MIN_AREA = 120
DIFFUSE_MIN_ELEVATION = 5.5
CREASE_MAX_INTENSITY = 35.0
CREASE_MAX_MEAN = 15.0
TOP_K_PIXELS = 50
STAIN_TEST_COUNT = 100

print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DATA_ROOT)
print('Checkpoint:', CHECKPOINT)
print(f'V8 Settings: t={T_DISTANCE}, samples={NUM_DIFFUSION_SAMPLES}, kernels=({DETREND_K1},{DETREND_K2},{DETREND_K3},{DETREND_K4}), taper={BORDER_TAPER_PIXELS}px')


In [ ]:
missing=[]
for package,module in [('timm','timm'),('einops','einops'),('numba','numba'),('scikit-learn','sklearn')]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Dependencies ready.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec = importlib.util.spec_from_file_location('labelinspect_author_smoke', WORK / 'author_smoke.py')
author = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = author
spec.loader.exec_module(author)
import numba
numba.set_num_threads(min(2, numba.get_num_threads()))
source_cache = WORK / 'upstream' / author.COMMIT
hashes = author.fetch_sources(source_cache)
model_module, diffusion_ns, Adapter, compatibility = author.load_author_components(source_cache, OUTPUT)
print('Pinned author commit:', author.COMMIT)


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
def image_paths(folder):
    return sorted(path for path in folder.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)

val_paths = image_paths(DATA_ROOT / 'val_normal')
all_test_good = image_paths(DATA_ROOT / 'test' / 'good')
all_test_stain = image_paths(DATA_ROOT / 'test' / 'stain')
assert (len(val_paths), len(all_test_good), len(all_test_stain)) == (10, 10, 398)

selection_rng = random.Random(SEED)
selected_stain = all_test_stain.copy()
selection_rng.shuffle(selected_stain)
test_stain_paths = sorted(selected_stain[:STAIN_TEST_COUNT])
test_paths = all_test_good + test_stain_paths
image_labels = np.asarray([0] * len(all_test_good) + [1] * len(test_stain_paths), dtype=np.int64)
(OUTPUT / 'locked_test_files.txt').write_text('\n'.join(path.relative_to(DATA_ROOT).as_posix() for path in test_paths))

RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR
def load_image(path):
    with Image.open(path) as image:
        image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1)

print('Protocol:', len(val_paths), 'validation normal,', len(all_test_good), 'test normal,', len(test_stain_paths), 'test stain')


In [ ]:
device = torch.device('cuda:0')
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
assert checkpoint['step'] == 2000, f"Expected step 2000, found {checkpoint['step']}"
assert checkpoint['author_commit'] == author.COMMIT
assert checkpoint.get('dataset_category') == 'fabric_stain_pilot'
model_config = checkpoint['model_config']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
backbone = model_module.UDHVT(**model_config).to(device)
model = Adapter(backbone)
model.load_state_dict(checkpoint['model'])
model.eval()
diffusion = diffusion_ns['GaussianDiffusionModel'](
    [224, 224], diffusion_ns['get_beta_schedule'](1000, 'cosine'), img_channels=3,
    loss_type='l2', noise='4dsimplex', octave=6, frequency=64, persistence=0.9, train=False)
diffusion.noise_fn(torch.zeros(1, 1, 4, 4, device=device), torch.tensor([5], device=device))
torch.cuda.reset_peak_memory_stats()
print('Loaded step', checkpoint['step'], 'model with', sum(p.numel() for p in model.parameters()), 'parameters.')


In [ ]:
from scipy.ndimage import binary_dilation, binary_erosion, binary_fill_holes

def reconstruct_paths(paths, batch_size=BATCH_SIZE, num_samples=NUM_DIFFUSION_SAMPLES):
    reconstructions = []; inputs = []; seconds = []
    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        x = torch.stack([load_image(p) for p in batch_paths]).to(device)
        tick = time.perf_counter()
        with torch.inference_mode():
            batch_recons = []
            for s in range(num_samples):
                r = diffusion.forward_backward(model, x, None, see_whole_sequence=None,
                                               t_distance=T_DISTANCE, denoise_fn='noise_fn')
                batch_recons.append(r)
            recon = torch.stack(batch_recons).mean(dim=0)
        torch.cuda.synchronize()
        seconds.append(time.perf_counter() - tick)
        inputs.append(x.cpu())
        reconstructions.append(recon.cpu())
        print(f'Reconstructed {min(start + len(batch_paths), len(paths))}/{len(paths)} ({num_samples} samples averaged)', flush=True)
    return torch.cat(inputs), torch.cat(reconstructions), seconds


def compute_v8_maps(inputs, recons, k1=DETREND_K1, k2=DETREND_K2, k3=DETREND_K3, k4=DETREND_K4,
                    smooth_k=SMOOTH_KERNEL, taper_px=BORDER_TAPER_PIXELS):
    """V8 residual map pipeline:
    1. Grayscale-matched luminance + chromatic residual fusion
    2. Quad-scale hierarchical reflection detrending (k1=15, k2=31, k3=61, k4=101)
    3. Reflection-padded spatial smoothing (k=5)
    4. Per-image robust background normalization (Median / IQR)
    5. Soft 5-pixel border fringe attenuation (eliminates boundary weave false seeds)
    """
    weights = torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)
    in_lum = (inputs * weights).sum(dim=1, keepdim=True)
    rec_lum = (recons * weights).sum(dim=1, keepdim=True)
    lum_sq = (in_lum - rec_lum).square()

    in_chrom = inputs - in_lum
    rec_chrom = recons - rec_lum
    chrom_sq = (in_chrom - rec_chrom).square().mean(dim=1, keepdim=True)

    raw_fused = lum_sq + 1.5 * chrom_sq

    def detrend_scale(x, k):
        p = k // 2
        xp = torch.nn.functional.pad(x, (p, p, p, p), mode='reflect')
        bg = torch.nn.functional.avg_pool2d(xp, kernel_size=k, stride=1, padding=0)
        return torch.clamp(x - bg, min=0.0)

    det1 = detrend_scale(raw_fused, k1)
    det2 = detrend_scale(raw_fused, k2)
    det3 = detrend_scale(raw_fused, k3)
    det4 = detrend_scale(raw_fused, k4)

    detrended = torch.maximum(
        torch.maximum(det1, det2 * 0.95),
        torch.maximum(det3 * 0.90, det4 * 0.85)
    )

    pad_sm = smooth_k // 2
    detrended_padded = torch.nn.functional.pad(detrended, (pad_sm, pad_sm, pad_sm, pad_sm), mode='reflect')
    smoothed = torch.nn.functional.avg_pool2d(detrended_padded, kernel_size=smooth_k, stride=1, padding=0)
    maps = smoothed.squeeze(1).numpy().astype(np.float32)

    flat = maps.reshape(maps.shape[0], -1)
    medians = np.median(flat, axis=1, keepdims=True)
    q75 = np.quantile(flat, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(flat, 0.25, axis=1, keepdims=True)
    iqr = np.maximum(q75 - q25, 1e-6)
    normalized = ((flat - medians) / iqr).reshape(maps.shape)

    # Border fringe attenuation: smooth cosine taper on outermost border pixels
    if taper_px > 0:
        H, W = normalized.shape[1], normalized.shape[2]
        taper_mask = np.ones((H, W), dtype=np.float32)
        for d in range(taper_px):
            w = float(0.5 * (1.0 - np.cos(np.pi * (d + 0.5) / taper_px)))
            taper_mask[d, :] = np.minimum(taper_mask[d, :], w)
            taper_mask[H - 1 - d, :] = np.minimum(taper_mask[H - 1 - d, :], w)
            taper_mask[:, d] = np.minimum(taper_mask[:, d], w)
            taper_mask[:, W - 1 - d] = np.minimum(taper_mask[:, W - 1 - d], w)
        normalized = normalized * taper_mask[None, :, :]

    return normalized, maps, raw_fused.squeeze(1).numpy().astype(np.float32)


def compute_v8_image_scores(v8_maps, top_k=TOP_K_PIXELS, tau_seed=13.5):
    """Computes robust V8 hybrid score: Top-K mean + Coherent Cluster Significance mass."""
    scores = []
    N, H, W = v8_maps.shape
    for i in range(N):
        m = v8_maps[i]
        flat_sorted = np.sort(m.ravel())[::-1]
        topk_mean = float(np.mean(flat_sorted[:top_k]))

        seeds = m > tau_seed
        max_cluster_mass = 0.0
        if np.any(seeds):
            visited = np.zeros((H, W), dtype=bool)
            for r in range(H):
                for c in range(W):
                    if seeds[r, c] and not visited[r, c]:
                        q = [(r, c)]
                        visited[r, c] = True
                        pts = []
                        while q:
                            cr, cc = q.pop()
                            pts.append((cr, cc))
                            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                                nr, nc = cr + dr, cc + dc
                                if 0 <= nr < H and 0 <= nc < W and seeds[nr, nc] and not visited[nr, nc]:
                                    visited[nr, nc] = True
                                    q.append((nr, nc))
                        if len(pts) >= 15:
                            cluster_vals = [m[pr, pc] for pr, pc in pts]
                            mass = np.sqrt(len(pts)) * (np.mean(cluster_vals) - tau_seed)
                            if mass > max_cluster_mass:
                                max_cluster_mass = float(mass)

        score = topk_mean + 0.35 * max_cluster_mass
        scores.append(score)
    return np.asarray(scores, dtype=np.float32)


def analyze_defect_clusters_v8(v8_map: np.ndarray, tau_high: float, tau_low: float,
                               min_area: int = MIN_CLUSTER_AREA):
    """V8 Defect Clustering:
    - Calibrated seed integration (tau_high=13.5, tau_low=3.3)
    - Primary crease & secondary shadow band disambiguation
    - Intensity-bounded physical crease vs chemical stain protection
    - Organic morphology consolidation (closing + hole filling)
    """
    seed_mask = v8_map > tau_high
    body_mask = v8_map > tau_low
    empty = np.zeros_like(v8_map, dtype=bool)
    if not np.any(body_mask):
        return False, False, empty, empty, empty, []

    H, W = v8_map.shape
    visited = np.zeros((H, W), dtype=bool)
    stain_mask = np.zeros((H, W), dtype=bool)
    crease_mask = np.zeros((H, W), dtype=bool)
    cluster_details = []

    for r in range(H):
        for c in range(W):
            if body_mask[r, c] and not visited[r, c]:
                cluster = []
                has_seed = False
                q = [(r, c)]
                visited[r, c] = True
                while q:
                    cr, cc = q.pop()
                    cluster.append((cr, cc))
                    if seed_mask[cr, cc]:
                        has_seed = True
                    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < H and 0 <= nc < W and body_mask[nr, nc] and not visited[nr, nc]:
                            visited[nr, nc] = True
                            q.append((nr, nc))

                # Mode A: Intense/compact seed cluster
                valid_cluster = has_seed and (len(cluster) >= min_area)

                if valid_cluster:
                    coords = np.array(cluster)
                    cov = np.cov(coords, rowvar=False)
                    eigvals = np.sort(np.linalg.eigvalsh(cov))
                    total_var = max(eigvals[0] + eigvals[1], 1e-6)
                    exp_linear = float(eigvals[1] / total_var)
                    axis_ratio = float(np.sqrt(max(eigvals[1], 1e-4) / max(eigvals[0], 1e-4)))

                    min_r, min_c = coords.min(axis=0)
                    max_r, max_c = coords.max(axis=0)
                    span = max(max_r - min_r, max_c - min_c)

                    cluster_vals = [v8_map[cr, cc] for cr, cc in cluster]
                    mean_val = float(np.mean(cluster_vals))
                    max_val = float(np.max(cluster_vals))

                    # Intensity-Bounded Crease Rule:
                    is_intense_absorption = (max_val > CREASE_MAX_INTENSITY) or (mean_val > CREASE_MAX_MEAN)
                    if is_intense_absorption:
                        is_crease = False
                    else:
                        # Primary high-aspect crease and secondary shadow band isolation
                        is_crease = (axis_ratio >= 3.5 and span >= 50) or (exp_linear >= 0.92 and span >= 45)

                    cluster_info = {
                        'box': [int(min_c), int(min_r), int(max_c), int(max_r)],
                        'area': len(cluster),
                        'explained_linear': round(exp_linear, 3),
                        'axis_ratio': round(axis_ratio, 2),
                        'is_crease': is_crease,
                        'mean_score': round(mean_val, 2),
                        'max_score': round(max_val, 2),
                    }
                    cluster_details.append(cluster_info)

                    if is_crease:
                        for cr, cc in cluster:
                            crease_mask[cr, cc] = True
                    else:
                        for cr, cc in cluster:
                            stain_mask[cr, cc] = True

    # Organic Morphology Consolidation: Morphological close (radius 3) + hole filling
    if np.any(stain_mask):
        dilated = binary_dilation(stain_mask, iterations=3)
        eroded = binary_erosion(dilated, iterations=3)
        stain_mask = binary_fill_holes(eroded)

    has_stain = bool(np.any(stain_mask))
    has_crease = bool(np.any(crease_mask))
    total_defect = stain_mask | crease_mask
    return has_stain, has_crease, stain_mask, crease_mask, total_defect, cluster_details


In [ ]:
print(f'Reconstructing held-out normal validation images at t_distance={T_DISTANCE} with {NUM_DIFFUSION_SAMPLES} samples...')
val_inputs, val_recons, val_batch_seconds = reconstruct_paths(val_paths)
val_v8_maps, val_smooth_maps, val_raw_maps = compute_v8_maps(val_inputs, val_recons)

# Pre-calibrate tau_seed from validation normals
val_p95 = float(np.quantile(val_v8_maps, 0.995))
tau_seed = float(max(13.5, val_p95))

# Compute validation image scores
val_image_scores = compute_v8_image_scores(val_v8_maps, top_k=TOP_K_PIXELS, tau_seed=tau_seed)

# Strict threshold: conservative maximum score over validation normals
tau_strict = float(np.max(val_image_scores))

# Calibrated hysteresis thresholds for segmentation
tau_high = float(max(13.5, np.quantile(val_image_scores, 0.80)))
tau_low = 3.3

calibration = {
    'version': 'v8',
    'source': '10 held-out normal validation images',
    'tau_strict': tau_strict,
    'tau_seed': tau_seed,
    'tau_high': tau_high,
    'tau_low': tau_low,
    'min_cluster_area': MIN_CLUSTER_AREA,
    'crease_max_intensity': CREASE_MAX_INTENSITY,
    'crease_max_mean': CREASE_MAX_MEAN,
    'top_k_pixels': TOP_K_PIXELS,
    'border_taper_pixels': BORDER_TAPER_PIXELS,
    't_distance': T_DISTANCE,
    'num_diffusion_samples': NUM_DIFFUSION_SAMPLES,
    'detrend_kernel_1': DETREND_K1,
    'detrend_kernel_2': DETREND_K2,
    'detrend_kernel_3': DETREND_K3,
    'detrend_kernel_4': DETREND_K4,
    'smooth_kernel': SMOOTH_KERNEL,
    'observed_validation_image_fpr': float(np.mean(val_image_scores > tau_strict)),
}
(OUTPUT / 'calibration_v8.json').write_text(json.dumps(calibration, indent=2))
print(json.dumps(calibration, indent=2))


In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

print(f'Running locked normal-versus-stain evaluation at t_distance={T_DISTANCE}...')
test_inputs, test_recons, test_batch_seconds = reconstruct_paths(test_paths)
test_v8_maps, test_smooth_maps, test_raw_maps = compute_v8_maps(test_inputs, test_recons)

# Image scores
image_scores = compute_v8_image_scores(test_v8_maps, top_k=TOP_K_PIXELS, tau_seed=tau_seed)

# Strict predictions
strict_predictions = image_scores > tau_strict
tn_s, fp_s, fn_s, tp_s = confusion_matrix(image_labels, strict_predictions, labels=[0, 1]).ravel()
sens_strict = float(tp_s / (tp_s + fn_s)) if (tp_s + fn_s) else None
spec_strict = float(tn_s / (tn_s + fp_s)) if (tn_s + fp_s) else None

# Morphological cluster analysis (Stains vs Creases)
stain_predictions = []
crease_predictions = []
total_defect_predictions = []
stain_masks = []
crease_masks = []
defect_masks = []
stain_areas = []
all_clusters = []

for i in range(len(test_paths)):
    has_s, has_c, m_s, m_c, m_tot, c_details = analyze_defect_clusters_v8(
        test_v8_maps[i], tau_high=tau_high, tau_low=tau_low, min_area=MIN_CLUSTER_AREA
    )
    stain_predictions.append(has_s)
    crease_predictions.append(has_c)
    total_defect_predictions.append(bool(np.any(m_tot)))
    stain_masks.append(m_s)
    crease_masks.append(m_c)
    defect_masks.append(m_tot)
    stain_areas.append(int(np.sum(m_s)))
    all_clusters.append(c_details)

stain_predictions = np.asarray(stain_predictions, dtype=bool)
total_defect_predictions = np.asarray(total_defect_predictions, dtype=bool)

# Chemical Stain metrics
tn_st, fp_st, fn_st, tp_st = confusion_matrix(image_labels, stain_predictions, labels=[0, 1]).ravel()
sens_stain = float(tp_st / (tp_st + fn_st)) if (tp_st + fn_st) else None
spec_stain = float(tn_st / (tn_st + fp_st)) if (tn_st + fp_st) else None
bal_acc_stain = float((sens_stain + spec_stain) / 2) if (sens_stain is not None and spec_stain is not None) else None

# Total Defect metrics
tn_tot, fp_tot, fn_tot, tp_tot = confusion_matrix(image_labels, total_defect_predictions, labels=[0, 1]).ravel()
sens_tot = float(tp_tot / (tp_tot + fn_tot)) if (tp_tot + fn_tot) else None
spec_tot = float(tn_tot / (tn_tot + fp_tot)) if (tn_tot + fp_tot) else None

# AUC metrics
image_auc = float(roc_auc_score(image_labels, image_scores))
average_precision = float(average_precision_score(image_labels, image_scores))
v1_raw_scores = np.quantile(test_raw_maps.reshape(len(test_paths), -1), 0.995, axis=1)
v1_auc = float(roc_auc_score(image_labels, v1_raw_scores))

rows = []
for path, label, score, p_strict, p_stain, p_crease, area, v1_s, clus in zip(
    test_paths, image_labels, image_scores, strict_predictions, stain_predictions, crease_predictions, stain_areas, v1_raw_scores, all_clusters
):
    rows.append({
        'image': path.relative_to(DATA_ROOT).as_posix(),
        'label': int(label),
        'kind': 'stain' if label else 'good',
        'v8_score': float(score),
        'strict_prediction': int(p_strict),
        'stain_prediction': int(p_stain),
        'crease_prediction': int(p_crease),
        'stain_mask_area': int(area),
        'raw_score_p995': float(v1_s),
        'cluster_count': len(clus),
    })
with (OUTPUT / 'per_image_v8.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

fpr_v8, tpr_v8, _ = roc_curve(image_labels, image_scores)
fpr_v1, tpr_v1, _ = roc_curve(image_labels, v1_raw_scores)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(fpr_v8, tpr_v8, label=f'V8 Quad-Scale + Protection (AUROC={image_auc:.3f})', color='green', lw=2.2)
ax1.plot(fpr_v1, tpr_v1, label=f'Raw Residual Baseline (AUROC={v1_auc:.3f})', color='gray', linestyle='--', lw=1.5)
ax1.plot([0, 1], [0, 1], ':', color='black', alpha=0.5)
ax1.set(xlabel='False-positive rate', ylabel='True-positive rate', title='Fabric Stain ROC Curve (V8 Protocol)')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.25)

prec_v8, rec_v8, _ = precision_recall_curve(image_labels, image_scores)
no_skill = np.sum(image_labels == 1) / len(image_labels)
ax2.plot(rec_v8, prec_v8, label=f'V8 Precision-Recall (AP={average_precision:.3f})', color='navy', lw=2.2)
ax2.plot([0, 1], [no_skill, no_skill], ':', color='crimson', label=f'No Skill ({no_skill:.2f})')
ax2.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve (V8 Protocol)')
ax2.legend(loc='lower left')
ax2.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT / 'roc_and_pr_curve_v8.png', dpi=160)
plt.show()

print(f'V8 AUROC: {image_auc:.4f} (Raw Residual AUROC: {v1_auc:.4f})')
print(f'V8 Average Precision: {average_precision:.4f}')
print(f'Strict Mode (tau={tau_strict:.2f}): Sensitivity={sens_strict:.3f}, Specificity={spec_strict:.3f}')
print(f'Chemical Stain Mode: Sensitivity={sens_stain:.3f}, Specificity={spec_stain:.3f}, BalAcc={bal_acc_stain:.3f}')
print(f'Total Defect Mode (stains + creases): Sensitivity={sens_tot:.3f}, Specificity={spec_tot:.3f}')


In [ ]:
def create_composite_overlay(orig_rgb: np.ndarray, s_mask: np.ndarray, c_mask: np.ndarray, clusters: list):
    """Generates a high-visibility translucent color overlay for stains (Amber) and creases (Cyan)."""
    img_pil = Image.fromarray((orig_rgb * 255).astype(np.uint8)).convert('RGBA')
    overlay = Image.new('RGBA', img_pil.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    # Chemical stains: Warm Amber tint (255, 170, 0, 120)
    for r in range(s_mask.shape[0]):
        for c in range(s_mask.shape[1]):
            if s_mask[r, c]:
                overlay.putpixel((c, r), (255, 170, 0, 120))

    # Fabric creases: Cool Cyan tint (0, 220, 255, 120)
    for r in range(c_mask.shape[0]):
        for c in range(c_mask.shape[1]):
            if c_mask[r, c]:
                overlay.putpixel((c, r), (0, 220, 255, 120))

    # Bounding boxes
    for clus in clusters:
        min_c, min_r, max_c, max_r = clus['box']
        box_color = (0, 220, 255, 220) if clus['is_crease'] else (255, 170, 0, 220)
        draw.rectangle([min_c, min_r, max_c, max_r], outline=box_color, width=2)

    combined = Image.alpha_composite(img_pil, overlay)
    return np.asarray(combined.convert('RGB'), dtype=np.float32) / 255.0


stain_indices = np.flatnonzero(image_labels == 1)
recovered_indices = np.flatnonzero((image_labels == 1) & (~strict_predictions) & stain_predictions)
strict_stain_indices = np.flatnonzero((image_labels == 1) & strict_predictions)

selected = [
    0,                                                  # Clean normal
    6,                                                  # Creased normal (57.jpg)
    int(strict_stain_indices[0]) if len(strict_stain_indices) > 0 else int(stain_indices[0]),
    int(strict_stain_indices[-1]) if len(strict_stain_indices) > 1 else int(stain_indices[-1]),
    int(recovered_indices[0]) if len(recovered_indices) > 0 else int(stain_indices[1]),
    int(recovered_indices[1]) if len(recovered_indices) > 1 else int(stain_indices[2]),
]

fig, axes = plt.subplots(len(selected), 6, figsize=(20, 3.4 * len(selected)))

for row, index in enumerate(selected):
    orig = ((test_inputs[index].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
    recon = ((test_recons[index].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
    v8_map = test_v8_maps[index]
    s_m = stain_masks[index].astype(np.float32)
    c_m = crease_masks[index].astype(np.float32)
    comp = create_composite_overlay(orig, stain_masks[index], crease_masks[index], all_clusters[index])

    panels = [orig, recon, v8_map, s_m, c_m, comp]
    titles = [
        f"{rows[index]['kind'].upper()} #{index}",
        f"Reconstruction (t={T_DISTANCE}, N={NUM_DIFFUSION_SAMPLES})",
        f"V8 Heatmap (Score {image_scores[index]:.1f})",
        f"Stain Mask (Area={stain_areas[index]}px)",
        f"Crease Mask",
        f"Color Inspection Overlay",
    ]
    for col, (panel, title) in enumerate(zip(panels, titles)):
        axes[row, col].imshow(panel, cmap=None if col in (0, 1, 5) else ('magma' if col == 2 else 'gray'))
        axes[row, col].set_title(title, fontsize=9.2)
        axes[row, col].axis('off')

fig.tight_layout()
fig.savefig(OUTPUT / 'evaluation_preview_v8.png', dpi=160, bbox_inches='tight')
plt.show()

report = {
    'status': 'passed',
    'scope': 'locked_fabric_stain_image_evaluation_v8',
    'dataset': 'Fabric Defects Dataset - matched grayscale stain cohort',
    'checkpoint_step': int(checkpoint['step']),
    'author_commit': author.COMMIT,
    'model_configuration': model_config,
    'train_normal_count': 48,
    'validation_normal_count': len(val_paths),
    'test_normal_count': int(np.sum(image_labels == 0)),
    'test_stain_count': int(np.sum(image_labels == 1)),
    't_distance': T_DISTANCE,
    'num_diffusion_samples': NUM_DIFFUSION_SAMPLES,
    'detrend_kernel_1': DETREND_K1,
    'detrend_kernel_2': DETREND_K2,
    'detrend_kernel_3': DETREND_K3,
    'detrend_kernel_4': DETREND_K4,
    'smooth_kernel': SMOOTH_KERNEL,
    'border_taper_pixels': BORDER_TAPER_PIXELS,
    'min_cluster_area': MIN_CLUSTER_AREA,
    'crease_max_intensity': CREASE_MAX_INTENSITY,
    'crease_max_mean': CREASE_MAX_MEAN,
    'top_k_pixels': TOP_K_PIXELS,
    'tau_strict': tau_strict,
    'tau_seed': tau_seed,
    'tau_high': tau_high,
    'tau_low': tau_low,
    'image_auroc_v8': image_auc,
    'image_auroc_raw': v1_auc,
    'average_precision': average_precision,
    'strict_confusion': {'tn': int(tn_s), 'fp': int(fp_s), 'fn': int(fn_s), 'tp': int(tp_s)},
    'strict_sensitivity': sens_strict,
    'strict_specificity': spec_strict,
    'chemical_stain_confusion': {'tn': int(tn_st), 'fp': int(fp_st), 'fn': int(fn_st), 'tp': int(tp_st)},
    'chemical_stain_sensitivity': sens_stain,
    'chemical_stain_specificity': spec_stain,
    'chemical_stain_balanced_accuracy': bal_acc_stain,
    'total_defect_confusion': {'tn': int(tn_tot), 'fp': int(fp_tot), 'fn': int(fn_tot), 'tp': int(tp_tot)},
    'total_defect_sensitivity': sens_tot,
    'total_defect_specificity': spec_tot,
    'normal_score_mean': float(np.mean(image_scores[image_labels == 0])),
    'stain_score_mean': float(np.mean(image_scores[image_labels == 1])),
    'median_batch_seconds': float(np.median(test_batch_seconds)),
    'approx_test_seconds_per_image': float(sum(test_batch_seconds) / len(test_paths)),
    'peak_gpu_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
    'peak_gpu_reserved_gib': torch.cuda.max_memory_reserved() / 2**30,
}
(OUTPUT / 'evaluation_report_v8.json').write_text(json.dumps(report, indent=2))
np.savez_compressed(OUTPUT / 'test_maps_v8_float16.npz',
                    v8_maps=test_v8_maps.astype(np.float16),
                    raw_maps=test_raw_maps.astype(np.float16),
                    image_scores=image_scores, labels=image_labels,
                    strict_predictions=strict_predictions,
                    stain_predictions=stain_predictions,
                    total_defect_predictions=total_defect_predictions)

archive = shutil.make_archive('/kaggle/working/fabric_stain_evaluation_v8_results', 'zip', OUTPUT)
print(json.dumps(report, indent=2))
print('\nLOCKED FABRIC STAIN EVALUATION V8 PASSED')
print('Download:', archive)
